#### Early Stoppping on HR data

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_selector, ColumnTransformer
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler


In [2]:
hr=pd.read_csv("D:\Machine Learning\Datasets\HR_comma_sep.csv")
X , y = hr.drop('left',axis=1) , hr['left']
ohe = OneHotEncoder(drop='first', sparse_output=False).set_output(transform="pandas")
col_transformer = ColumnTransformer([("OHE", ohe, make_column_selector(dtype_include=object))], 
                                    remainder='passthrough', 
                                    verbose_feature_names_out=False)
col_transformer = col_transformer.set_output(transform="pandas")
X = col_transformer.fit_transform(X)
X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.3,random_state=25,stratify=y)
scl_x = MinMaxScaler()
X_train_scaled = scl_x.fit_transform(X_train)
X_test_scaled = scl_x.transform(X_test)

#### Early Stopping

In [12]:
from tensorflow.keras.callbacks import EarlyStopping
initializer = tf.keras.initializers.GlorotUniform(seed= 25)
tf.random.set_seed(seed=25)
early = EarlyStopping(monitor='val_loss', min_delta=0.001, patience=5, verbose=2,restore_best_weights=True)
model = tf.keras.Sequential([tf.keras.Input(shape=(X_train_scaled.shape[1],)),
                            tf.keras.layers.Dense(14,activation='relu', kernel_initializer=initializer),
                            tf.keras.layers.Dense(9,activation='relu', kernel_initializer=initializer),
                            tf.keras.layers.Dense(4,activation='relu', kernel_initializer=initializer),
                            tf.keras.layers.Dense(1,activation='sigmoid', kernel_initializer=initializer)]) # first parameter → no. layers
model.compile(optimizer= tf.keras.optimizers.RMSprop(learning_rate=0.001),
              loss=tf.keras.losses.BinaryCrossentropy())
history = model.fit(X_train_scaled, y_train, validation_data=(X_test_scaled, y_test),callbacks=[early], epochs= 500)

Epoch 1/500
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5966 - val_loss: 0.5241
Epoch 2/500
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5005 - val_loss: 0.4716
Epoch 3/500
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4410 - val_loss: 0.4071
Epoch 4/500
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3879 - val_loss: 0.3585
Epoch 5/500
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3428 - val_loss: 0.3187
Epoch 6/500
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3081 - val_loss: 0.2920
Epoch 7/500
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.2852 - val_loss: 0.2738
Epoch 8/500
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.2689 - val_loss: 0.2603
Epoch 9/500
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.2564 - val_loss: 0.2497
Epoch 10/500
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.2462 - val_loss: 0.2405
Epoch 11/500
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.2369 - val_loss: 0.2324
Epoch 12/500
328/328 ━━━━━━━━━━━━━━━━━━━━

In [13]:
from sklearn.metrics import log_loss
y_pred_prob = model.predict(X_test_scaled)
print(log_loss(y_test, y_pred_prob))
      

141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
0.16058233057127397


In [14]:
from sklearn.metrics import accuracy_score
y_pred = np.where(y_pred_prob >= 0.5,1,0)
print(accuracy_score(y_test, y_pred))

0.9517670593465215


#### Reduce LR on Plateau

In [15]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

In [22]:
initializer = tf.keras.initializers.GlorotUniform(seed= 25)
tf.random.set_seed(seed=25)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor= 0.2, patience=5,min_lr=0.001)
model = tf.keras.Sequential([tf.keras.Input(shape=(X_train_scaled.shape[1],)),
                            tf.keras.layers.Dense(14,activation='relu', kernel_initializer=initializer),
                            tf.keras.layers.Dense(9,activation='relu', kernel_initializer=initializer),
                            tf.keras.layers.Dense(4,activation='relu', kernel_initializer=initializer),
                            tf.keras.layers.Dense(1,activation='sigmoid', kernel_initializer=initializer)]) # first parameter → no. layers
model.compile(optimizer= tf.keras.optimizers.RMSprop(learning_rate=0.001),
              loss=tf.keras.losses.BinaryCrossentropy())
history = model.fit(X_train_scaled, y_train, validation_data=(X_test_scaled, y_test),callbacks=[reduce_lr], epochs= 200)

Epoch 1/200
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5966 - val_loss: 0.5241 - learning_rate: 0.0010
Epoch 2/200
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.5005 - val_loss: 0.4716 - learning_rate: 0.0010
Epoch 3/200
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4410 - val_loss: 0.4071 - learning_rate: 0.0010
Epoch 4/200
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3879 - val_loss: 0.3585 - learning_rate: 0.0010
Epoch 5/200
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3428 - val_loss: 0.3187 - learning_rate: 0.0010
Epoch 6/200
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.3081 - val_loss: 0.2920 - learning_rate: 0.0010
Epoch 7/200
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.2852 - val_loss: 0.2738 - learning_rate: 0.0010
Epoch 8/200
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.2689 - val_loss: 0.2603 - learning_rate: 0.0010
Epoch 9/200
328/328 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.2564 - val_loss: 0.2497 - learning_rate: 0.0010
E

In [23]:
y_pred_prob = model.predict(X_test_scaled)
print(log_loss(y_test, y_pred_prob))

141/141 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
0.153788619118786


In [24]:
y_pred = np.where(y_pred_prob >= 0.5,1,0)
print(accuracy_score(y_test, y_pred))

0.9575461213603023
